In [2]:
import sys, time, importlib
from pathlib import Path
import duckdb, pandas as pd

ROOT = Path.cwd().parent
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
import src.blocking as B

ARTIFACTS = ROOT / "artifacts"
PARQ, KEYS2 = ARTIFACTS / "parquet", ARTIFACTS / "keys_v2"
VALID_GT = ROOT / "notebooks" / "phase2_valid_gt.parquet"
def pq(name): return f"read_parquet('{(PARQ / f'{name}.parquet').as_posix()}')"
def k2(name): return f"read_parquet('{(KEYS2 / f'{name}.parquet').as_posix()}')"

con = duckdb.connect((ARTIFACTS / "blocking.duckdb").as_posix())
B.setup(con, threads=4, memory_limit="4GB", temp_dir=(ARTIFACTS / "duckdb_tmp").as_posix())

S1_ALL    = k2("train_source1")
TGT_PARTS = [k2("train_source2"), k2("train_source3")]
TGT       = "(" + " UNION ALL ".join(f"SELECT * FROM {p}" for p in TGT_PARTS) + ")"

print(con.execute("SELECT (SELECT count(*) FROM eval_s1), (SELECT count(*) FROM eval_pairs), (SELECT count(*) FROM cand2)").fetchone())
display(B.eval_blocking(con, "(SELECT s1_id, tgt_id FROM cand2 WHERE rk <= 5)", "cand2_top5/src"))

(20000, 69252, 1786478)


,method,pair_recall,s1_coverage,ceiling_f05,avg_cands,med_cands,p99_cands,max_cands
0,cand2_top5/src,0.939756,0.995563,0.981328,9.9206,10.0,10.0,10


In [3]:
con.execute(f"""
CREATE OR REPLACE TABLE train_s1 AS
SELECT eid(source1_entity_id) AS s1_id, matched_entity_ids
FROM (SELECT g.* FROM {pq('train_ground_truth')} g
      ANTI JOIN read_parquet('{VALID_GT.as_posix()}') v USING (source1_entity_id))
USING SAMPLE 30000 ROWS (reservoir, 11)
""")
con.execute("""
CREATE OR REPLACE TABLE train_pairs AS
SELECT s1_id, eid(trim(tgt)) AS tgt_id
FROM (SELECT s1_id, unnest(string_split(matched_entity_ids, ',')) AS tgt FROM train_s1
      WHERE matched_entity_ids IS NOT NULL AND trim(matched_entity_ids) <> '')
""")
print(con.execute("SELECT (SELECT count(*) FROM train_s1), (SELECT count(*) FROM train_pairs)").fetchone())

B.run_blocking(con, "cand_tr", S1_ALL, TGT_PARTS, s1_ids_tbl="train_s1")
display(B.eval_blocking(con, "(SELECT s1_id, tgt_id FROM cand_tr WHERE rk <= 5)", "train_top5/src",
                        s1_tbl="train_s1", pairs_tbl="train_pairs"))

(30000, 104127)


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

   cand_tr india         87s


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

   cand_tr us           183s


,method,pair_recall,s1_coverage,ceiling_f05,avg_cands,med_cands,p99_cands,max_cands
0,train_top5/src,0.940467,0.995234,0.981208,9.923033,10.0,10.0,10


In [4]:
print(con.execute(f"""
SELECT count(*) FILTER (WHERE name_v2 LIKE '% com' OR name_v2 LIKE 'www %') AS web_tokens_left,
       count(*) FILTER (WHERE regexp_matches(addr_v2, '(^| )0[0-9]'))    AS leading_zero_nums
FROM {k2('train_source2')}
""").fetchone())

(199081, 124930)


In [5]:
  B.run_blocking(con, "cand2", S1_ALL, TGT_PARTS, s1_ids_tbl="eval_s1")
  display(pd.concat([B.eval_blocking(con, f"(SELECT s1_id, tgt_id FROM cand2 WHERE rk <= {k})", f"v3_top{k}/src")
                     for k in (1, 3, 5, 10, 20, 50)], ignore_index=True))

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

   cand2 us            96s


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

   cand2 india        182s


,method,pair_recall,s1_coverage,ceiling_f05,avg_cands,med_cands,p99_cands,max_cands
0,v3_top1/src,0.450803,0.977761,0.806219,1.99885,2.0,2.0,2
1,v3_top3/src,0.867282,0.994823,0.966870,5.97940,6.0,6.0,6
2,v3_top5/src,0.939742,0.995563,0.981323,9.92060,10.0,10.0,10
3,v3_top10/src,0.956564,0.996514,0.985948,19.61145,20.0,20.0,20
4,v3_top20/src,0.964709,0.997200,0.988492,38.30785,40.0,40.0,40
5,v3_top50/src,0.972867,0.997887,0.991293,89.32390,100.0,100.0,100


In [6]:
import src.normalize as N, src.keys as KY
importlib.reload(N); importlib.reload(KY)
print("web tokens stripped:", N.normalize_name_v2("clyialsystems.com"))   # must print 'clyialsystems'

ALL = [f"{s}_source{i}" for s in ("train", "test") for i in (1, 2, 3)]
KY.build_all(con, PARQ, KEYS2, ALL, overwrite=True)      # ~25 min

B.run_blocking(con, "cand_tr", S1_ALL, TGT_PARTS, s1_ids_tbl="train_s1")
B.run_blocking(con, "cand2",   S1_ALL, TGT_PARTS, s1_ids_tbl="eval_s1")
display(pd.concat([B.eval_blocking(con, f"(SELECT s1_id, tgt_id FROM cand2 WHERE rk <= {k})", f"v3new_top{k}/src")
                   for k in (1, 3, 5, 10, 20, 50)], ignore_index=True))

web tokens stripped: clyialsystems


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

   keys train_source1      127.1s


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

   keys train_source2      295.5s


...

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

   cand2 us            95s


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

   cand2 india        182s


,method,pair_recall,s1_coverage,ceiling_f05,avg_cands,med_cands,p99_cands,max_cands
0,v3new_top1/src,0.453157,0.978765,0.808405,1.99895,2.0,2.0,2
1,v3new_top3/src,0.872307,0.994982,0.968491,5.97965,6.0,6.0,6
2,v3new_top5/src,0.945056,0.995827,0.982881,9.92195,10.0,10.0,10
3,v3new_top10/src,0.960795,0.996883,0.987277,19.61630,20.0,20.0,20
4,v3new_top20/src,0.968203,0.997359,0.989549,38.33360,40.0,40.0,40
5,v3new_top50/src,0.975625,0.998204,0.992219,89.42870,100.0,100.0,100


In [7]:
import lightgbm as lgb
from sklearn.metrics import roc_auc_score

def feat_sql(cand):
    return f"""
    WITH c AS (SELECT * FROM {cand}),
    s AS (SELECT eid(entity_id) AS id, name_core, addr_v2, postcode FROM {S1_ALL}
          WHERE eid(entity_id) IN (SELECT DISTINCT s1_id FROM c)),
    t AS (SELECT eid(entity_id) AS id, name_core, addr_v2, postcode FROM {TGT}
          WHERE eid(entity_id) IN (SELECT DISTINCT tgt_id FROM c)),
    j AS (
        SELECT c.s1_id, c.tgt_id, c.score, c.n_keys, c.rk, c.tgt_id // {B.SRC} AS src,
               coalesce(s.name_core, '') AS sn, coalesce(t.name_core, '') AS tn,
               t.addr_v2 AS ta, s.postcode AS sp, t.postcode AS tp,
               string_split(coalesce(s.name_core, ''), ' ') AS snt, string_split(coalesce(t.name_core, ''), ' ') AS tnt,
               string_split(coalesce(s.addr_v2, ''), ' ')   AS sat, string_split(coalesce(t.addr_v2, ''), ' ')   AS tat,
               regexp_extract_all(coalesce(s.addr_v2, ''), '\\d+') AS snum,
               regexp_extract_all(coalesce(t.addr_v2, ''), '\\d+') AS tnum
        FROM c JOIN s ON s.id = c.s1_id JOIN t ON t.id = c.tgt_id
    )
    SELECT s1_id, tgt_id, src, rk, score, n_keys,
           score / max(score) OVER (PARTITION BY s1_id, src)                        AS score_rel,
           jaro_winkler_similarity(sn, tn)                                          AS name_jw,
           jaro_winkler_similarity(replace(sn, ' ', ''), replace(tn, ' ', ''))      AS cmp_jw,
           CASE WHEN length(sn) > 2 AND length(tn) > 2
                     AND (contains(replace(tn, ' ', ''), replace(sn, ' ', ''))
                          OR contains(replace(sn, ' ', ''), replace(tn, ' ', ''))) THEN 1 ELSE 0 END AS cmp_contains,
           len(list_intersect(snt, tnt)) / greatest(len(list_distinct(list_concat(snt, tnt))), 1) AS name_jacc,
           len(list_intersect(sat, tat)) / greatest(len(list_distinct(list_concat(sat, tat))), 1) AS addr_jacc,
           len(list_intersect(snum, tnum))                                          AS num_shared,
           len(snum) AS s_nums, len(tnum) AS t_nums,
           (ta IS NULL)::INT                                                        AS t_addr_null,
           CASE WHEN sp IS NULL OR tp IS NULL THEN -1 WHEN sp = tp THEN 1 ELSE 0 END AS pc_match
    FROM j"""

FEATS = ["src", "rk", "score", "n_keys", "score_rel", "name_jw", "cmp_jw", "cmp_contains",
         "name_jacc", "addr_jacc", "num_shared", "s_nums", "t_nums", "t_addr_null", "pc_match"]

t0 = time.time()
tr_df = con.execute(f"""SELECT f.*, (p.tgt_id IS NOT NULL)::INT AS y
    FROM ({feat_sql('cand_tr')}) f LEFT JOIN train_pairs p USING (s1_id, tgt_id)""").fetchdf()
ev_df = con.execute(f"""SELECT f.*, (p.tgt_id IS NOT NULL)::INT AS y
    FROM ({feat_sql('cand2')}) f LEFT JOIN eval_pairs p USING (s1_id, tgt_id)""").fetchdf()
print(f"features: train {len(tr_df):,} rows (pos {tr_df.y.mean():.3f}) | eval {len(ev_df):,}  ({time.time() - t0:.0f}s)")

rr = lgb.LGBMClassifier(n_estimators=400, learning_rate=0.05, num_leaves=63, min_child_samples=50,
                        subsample=0.8, subsample_freq=1, colsample_bytree=0.8, verbose=-1)
rr.fit(tr_df[FEATS], tr_df["y"])
ev_df["p"] = rr.predict_proba(ev_df[FEATS])[:, 1]
print(f"eval AUC: {roc_auc_score(ev_df.y, ev_df.p):.4f}")
display(pd.Series(rr.feature_importances_, index=FEATS).sort_values(ascending=False).to_frame("importance").T)

con.register("ev_scored", ev_df[["s1_id", "tgt_id", "src", "p"]])
con.execute("""CREATE OR REPLACE TABLE cand_rr AS
    SELECT *, row_number() OVER (PARTITION BY s1_id, src ORDER BY p DESC, tgt_id) AS rk2 FROM ev_scored""")

rows  = [B.eval_blocking(con, f"(SELECT s1_id, tgt_id FROM cand2 WHERE rk <= {k})", f"keyscore_top{k}/src") for k in (3, 5)]
rows += [B.eval_blocking(con, f"(SELECT s1_id, tgt_id FROM cand_rr WHERE rk2 <= {k})", f"rerank_top{k}/src") for k in (1, 2, 3, 5)]
rows += [B.eval_blocking(con, f"(SELECT s1_id, tgt_id FROM cand_rr WHERE rk2 <= {cap} AND p >= {tau})",
                         f"rerank p>={tau} cap{cap}")
         for tau in (0.01, 0.03, 0.1) for cap in (3, 5)]
display(pd.concat(rows, ignore_index=True))

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

features: train 2,675,295 rows (pos 0.038) | eval 1,788,574  (52s)
eval AUC: 0.9990


,score,addr_jacc,cmp_jw,name_jw,score_rel,name_jacc,rk,t_nums,n_keys,num_shared,s_nums,cmp_contains,src,t_addr_null,pc_match
importance,3337,3217,3043,2892,2790,2048,1775,1256,1094,1000,959,548,353,286,202


,method,pair_recall,s1_coverage,ceiling_f05,avg_cands,med_cands,p99_cands,max_cands
0,keyscore_top3/src,0.872307,0.994982,0.968491,5.97965,6.0,6.0,6
1,keyscore_top5/src,0.945056,0.995827,0.982881,9.92195,10.0,10.0,10
2,rerank_top1/src,0.492159,0.995457,0.845027,1.99895,2.0,2.0,2
3,rerank_top2/src,0.789017,0.997940,0.955187,3.99410,4.0,4.0,4
4,rerank_top3/src,0.920638,0.998046,0.983308,5.97965,6.0,6.0,6
5,rerank_top5/src,0.972333,0.998151,0.991531,9.92195,10.0,10.0,10
6,rerank p>=0.01 cap3,0.919396,0.997993,0.982886,4.61830,5.0,6.0,6
7,rerank p>=0.01 cap5,0.969618,0.998098,0.990801,5.48855,5.0,10.0,10
8,rerank p>=0.03 cap3,0.917345,0.997781,0.982107,4.25125,4.0,6.0,6
9,rerank p>=0.03 cap5,0.966052,0.997834,0.989675,4.79645,5.0,10.0,10


In [8]:
def eval_f05(con, pred, label, s1_tbl="eval_s1", pairs_tbl="eval_pairs"):
    """Official metric: per-S1 F0.5 (singleton + empty = 1, any wrong/empty otherwise = 0), macro-averaged."""
    return con.execute(f"""
    WITH pr AS (SELECT DISTINCT s1_id, tgt_id FROM {pred}),
         nt AS (SELECT s1_id, count(*) AS nt FROM {pairs_tbl} GROUP BY 1),
         np AS (SELECT s1_id, count(*) AS np FROM pr GROUP BY 1),
         tp AS (SELECT p.s1_id, count(*) AS tp FROM pr p JOIN {pairs_tbl} g USING (s1_id, tgt_id) GROUP BY 1),
         per AS (SELECT e.s1_id, coalesce(nt, 0) nt, coalesce(np, 0) np, coalesce(tp, 0) tp
                 FROM {s1_tbl} e LEFT JOIN nt USING (s1_id) LEFT JOIN np USING (s1_id) LEFT JOIN tp USING (s1_id)),
         f AS (SELECT *, CASE WHEN nt = 0 AND np = 0 THEN 1.0
                              WHEN tp = 0 THEN 0.0
                              ELSE 1.25 * (tp / np) * (tp / nt) / (0.25 * (tp / np) + (tp / nt)) END AS f05
               FROM per)
    SELECT '{label}' AS method,
           avg(f05)                                   AS macro_f05,
           avg(CASE WHEN nt = 0 THEN f05 END)         AS f05_singletons,
           avg(CASE WHEN nt > 0 THEN f05 END)         AS f05_matched,
           sum(tp) / greatest(sum(np), 1)             AS micro_precision,
           sum(tp) / sum(nt)                          AS micro_recall,
           avg(np)                                    AS avg_pred
    FROM f""").fetchdf()

CUT = "(SELECT * FROM cand_rr WHERE rk2 <= 5 AND p >= 0.01)"       # chosen candidate set
rows = [eval_f05(con, f"(SELECT s1_id, tgt_id FROM {CUT} WHERE p >= {t})", f"rerank-as-matcher p>={t}")
        for t in (0.2, 0.3, 0.4, 0.5, 0.6, 0.7, 0.8, 0.9)]
display(pd.concat(rows, ignore_index=True))

,method,macro_f05,f05_singletons,f05_matched,micro_precision,micro_recall,avg_pred
0,rerank-as-matcher p>=0.2,0.876657,0.446211,0.900963,0.885649,0.940334,3.67640
1,rerank-as-matcher p>=0.3,0.897430,0.533209,0.917997,0.914737,0.928580,3.51500
2,rerank-as-matcher p>=0.4,0.908735,0.597755,0.926296,0.932728,0.917374,3.40560
3,rerank-as-matcher p>=0.5,0.917749,0.673527,0.931539,0.946831,0.905158,3.31020
4,rerank-as-matcher p>=0.6,0.927541,0.811974,0.934067,0.964553,0.883296,3.17090
5,rerank-as-matcher p>=0.7,0.928972,0.872778,0.932146,0.973808,0.867051,3.08300
6,rerank-as-matcher p>=0.8,0.924582,0.913938,0.925183,0.982033,0.844495,2.97765
7,rerank-as-matcher p>=0.9,0.912019,0.953227,0.909692,0.989053,0.806258,2.82265


In [9]:
import numpy as np

cands = con.execute(f"SELECT s1_id, tgt_id, p FROM {CUT}").fetchdf()
cands = cands.sort_values(["s1_id", "p"], ascending=[True, False]).reset_index(drop=True)

def expected_f05_decide(df, miss=0.0, a=1.0):
    d = df.copy()
    d["q"] = d["p"].clip(1e-9, 1 - 1e-9) ** a
    g = d.groupby("s1_id", sort=False)
    d["k"]     = g.cumcount() + 1
    d["S_k"]   = g["q"].cumsum()
    d["S_all"] = g["q"].transform("sum") + miss
    d["ef"]    = 1.25 * d["S_k"] / (0.25 * d["S_all"] + d["k"])
    d["l1m"]   = np.log1p(-d["q"])
    e0 = np.exp(g["l1m"].transform("sum") - miss)                 # P(no true match at all)
    d["e0"] = e0
    best = d.loc[d.groupby("s1_id", sort=False)["ef"].idxmax(), ["s1_id", "k", "ef", "e0"]]
    best["k_star"] = np.where(best["ef"] > best["e0"], best["k"], 0)
    d = d.merge(best[["s1_id", "k_star"]], on="s1_id")
    return d.loc[d["k"] <= d["k_star"], ["s1_id", "tgt_id"]]

rows = [eval_f05(con, "(SELECT s1_id, tgt_id FROM cand_rr WHERE rk2 <= 5 AND p >= 0.01 AND p >= 0.7)",
                 "global p>=0.7 (baseline)")]
for miss in (0.0, 0.1, 0.3):
    for a in (1.0, 1.5, 2.0):
        pred = expected_f05_decide(cands, miss=miss, a=a)
        con.register("pred_ef", pred)
        rows.append(eval_f05(con, "(SELECT s1_id, tgt_id FROM pred_ef)", f"expF05 miss={miss} a={a}"))
display(pd.concat(rows, ignore_index=True))

,method,macro_f05,f05_singletons,f05_matched,micro_precision,micro_recall,avg_pred
0,global p>=0.7 (baseline),0.928972,0.872778,0.932146,0.973808,0.867051,3.08300
1,expF05 miss=0.0 a=1.0,0.921540,0.601497,0.939613,0.967880,0.870242,3.11330
2,expF05 miss=0.0 a=1.5,0.927341,0.787652,0.935229,0.978321,0.850387,3.00980
3,expF05 miss=0.0 a=2.0,0.925586,0.847521,0.929995,0.982602,0.835095,2.94280
4,expF05 miss=0.1 a=1.0,0.921335,0.594013,0.939819,0.967327,0.871267,3.11875
5,expF05 miss=0.1 a=1.5,0.926933,0.773620,0.935591,0.977620,0.851557,3.01610
6,expF05 miss=0.1 a=2.0,0.925735,0.842844,0.930415,0.982175,0.836236,2.94810
7,expF05 miss=0.3 a=1.0,0.920552,0.574369,0.940101,0.965862,0.873072,3.12995
8,expF05 miss=0.3 a=1.5,0.925314,0.726848,0.936521,0.976095,0.853780,3.02870
9,expF05 miss=0.3 a=2.0,0.926094,0.825070,0.931799,0.981559,0.838532,2.95805


In [10]:
import src.blocking as B, src.rerank as R
importlib.reload(B); importlib.reload(R)

RR_PATH = ARTIFACTS / "rerank_lgb.txt"
rr.booster_.save_model(str(RR_PATH))            # reranker trained on train_s1 (30k, no validation)
booster = R.load(RR_PATH)

con.execute("CREATE OR REPLACE TABLE cut_full (s1_id BIGINT, tgt_id BIGINT, src BIGINT, p DOUBLE, rk2 BIGINT)")
t0 = time.time()
B.run_blocking(con, "cut_full", S1_ALL, TGT_PARTS, s1_ids_tbl=None, k_max=30, s1_chunks=32,
               on_chunk=lambda tbl: R.rerank_and_cut(con, tbl, S1_ALL, TGT, booster, "cut_full", tau=0.01, cap=5))
print(f"done in {(time.time() - t0) / 60:.1f} min")

print(con.execute("SELECT count(*), count(DISTINCT s1_id), count(*) / count(DISTINCT s1_id) FROM cut_full").fetchone())
display(B.eval_blocking(con, "(SELECT s1_id, tgt_id FROM cut_full)", "full-run cut, eval S1"))

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

   cut_full us       S1 keys           42s


OutOfMemoryException: Out of Memory Error: Allocation failure